Imports:

In [1]:
import aerosandbox as asb
import aerosandbox.numpy as np
import casadi as ca

from wing_deflection import max_deflection

Constants, definitions that shouldn't change

In [2]:
wing_airfoil = asb.Airfoil("naca0001")
tail_airfoil = asb.Airfoil("naca0001")

E = 3e9
rho_balsa = 250 # kg/m3
rho = 1.225
nu = 1.5e-5
pi = np.pi
AR_v = 3


Beginning the optimization environment, initializing some variables


In [3]:
opti = asb.Opti()

V = opti.variable(init_guess = 4, lower_bound = 0.1, upper_bound = 7)
AR = opti.variable(init_guess = 6, lower_bound = 5, upper_bound = 15)
S = opti.variable(init_guess = 0.05, lower_bound = 0.01, upper_bound = 0.5)
twist = opti.variable(init_guess = 2, lower_bound = 0, upper_bound = 10)
dihedral = opti.variable(init_guess = 5, lower_bound = 0, upper_bound=10)
AR_h = opti.variable(init_guess = 4, lower_bound = 2, upper_bound =10)
S_h = opti.variable(init_guess = 0.01, lower_bound = 0.00001)
S_v = opti.variable(init_guess = 0.005, lower_bound = 0.000001)
glide_angle = opti.variable(init_guess = 5, lower_bound=0, upper_bound = 15)
H_loc = opti.variable(init_guess = - 0.2, upper_bound=-0) #horizontal stabilizer is at H_loc
H_loc_v = 0


Derived variables from the given optimized variables

In [4]:
b_total = ca.sqrt(S * AR)
c = S / b_total
b = b_total / 2

b_h = ca.sqrt(S_h * AR_h) / 2
c_h = S_h / (b_h * 2)

b_v = ca.sqrt(S_v * AR_v)
c_v = S_v / b_v

Weight and structural model

In [5]:
thickness = 0.002
boom_area = 0.01 * 0.01
margin = 0.01
I_formula = lambda chord, tau: chord * thickness ** 3 / 12

weight = (S + S_h + S_v) * thickness * rho_balsa - H_loc * rho_balsa * boom_area + margin

cumsum_weight = (c / 2 * S + (H_loc + c_h/2) * S_h + (H_loc_v + c_v/2) * S_v) * thickness * rho_balsa - H_loc / 2 * H_loc * boom_area * rho_balsa + H_loc * margin
COM = cumsum_weight / weight

# Structural Deflection Constraint (tau set to 0.05 dynamically)
N = 1.2
opti.subject_to(max_deflection(N * weight * 9.81, AR, S, E, I_formula = I_formula, tau=0.002) < 0.1 * b)
opti.subject_to(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.002) < 0.1 * b_h)
# opti.subject_to(max_deflection(N * weight * 9.81, AR_v, S_v, E, I_formula = I_formula, tau=0.05) < 0.02)


MX(fabs(opti0_lam_g_19))

Aerodynamic Constraints

In [6]:
import casadi as ca

#tail volume coefficients

l_h = H_loc - c / 4 + c_h / 4
l_v = H_loc_v - c / 4 + c_v / 4

hor_vol_coef = - S_h * l_h / (S * c)
ver_vol_coef = - S_v * l_v / (S * c)

opti.subject_to(hor_vol_coef > 0.1)
opti.subject_to(hor_vol_coef < 0.3)

opti.subject_to(ver_vol_coef > 0.02)
opti.subject_to(ver_vol_coef < 0.05)

#neutral point
a_w = 2 * pi / (1 + 2 / AR)
a_h = 2 * pi / (1 + 2 / AR_h)

# np = c * (a_w / (4 * a_h) + hor_vol_coef * (1 + c / (4 * l_h))) / (a_w / a_h + hor_vol_coef * c / l_h )
np = c * (S * a_w / (4 * S_h * a_h) + l_h / c + 0.25) / (S * a_w / (S_h * a_h) + 1)

#recalculate neutral point and stuff
opti.subject_to(np > COM)

#aerodynamic constraints

a0 = 2 * pi
e = 0.9
Re = V * c / nu

CL = a0 / (1 + a0 / (pi * AR * e)) * twist

CL_h = - (COM / c - 0.25) * CL / ((COM / H_loc - 1 - c / H_loc) * hor_vol_coef)
alpha_t = CL_h / (2 * pi)

CDi = CL ** 2 / (pi * AR * e) + CL_h ** 2 / (pi * AR_h * e)
CD0 = 1.3 * 2.656 / Re ** 0.5
CD = CDi + CD0

q = 1/2 * rho * V**2 * S
q_h = 1/2 * rho * V**2 * S_h
L = q * (CL) + q_h * CL_h
D = q * CD

opti.subject_to(weight * 9.81 * ca.cos(glide_angle * pi / 180) < L)
opti.subject_to(weight * 9.81 * ca.cos(glide_angle * pi / 180) > L - 0.01)

opti.subject_to(weight * 9.81 * ca.sin(glide_angle * pi / 180) > D)
opti.subject_to(weight * 9.81 * ca.sin(glide_angle * pi / 180) < D + 0.01)

opti.subject_to(CL < 1.4)
opti.subject_to(CL_h < 1.4)

#spiral stability
spiral  = - (COM - c_v / 4) * dihedral / (b_total * CL)

opti.subject_to(spiral > 4)

MX(fabs(opti0_lam_g_31))

Objective function and solving

In [7]:
total_dist = 1 / (ca.sin(glide_angle))


opti.maximize( total_dist / V)


# --- Solve the Problem ---
sol = opti.solve(behavior_on_failure="return_last", max_iter=10000,
    options={"ipopt.linear_solver": "mumps",
    "ipopt.mumps_mem_percent":500,
    })


This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:      102
Number of nonzeros in Lagrangian Hessian.............:       49

Total number of variables............................:       10
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:       31
        inequality constraints with only lower bounds:       12
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       19

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  2.6070880e-01 1.01e+02 9.55e-01   0.0 0.00e+00    -  0.00e+00 0.00e+00 

CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 18, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 21, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 21, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 21, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 18, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 18, col 0).") [.../casadi/core/oracle_function.cpp:408]
CasADi - 2026-07-15 08:10:47 WARNING("solver:nlp_g failed: NaN detected for output g, at (row 18, co

 108 -8.3008238e+02 1.42e-01 1.30e+11   3.1 2.90e+00   9.4 1.00e+00 3.61e-04h  5
 109 -2.0139548e+02 1.42e-01 1.61e+12   3.6 3.65e+00   9.8 1.68e-01 1.46e-03h  2
iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
 110 -2.2659726e+01 1.42e-01 7.74e+12   3.6 2.89e+01   9.3 1.00e-01 8.67e-04h  3
 111 -7.8017918e+00 1.41e-01 2.79e+13   3.6 5.05e+01   8.8 7.38e-01 1.13e-03h  4
 112 -5.2543505e+00 1.40e-01 2.69e+13   3.6 5.05e+01   8.4 5.21e-02 7.92e-03h  1
 113 -5.5975604e+00 1.40e-01 4.29e+13   3.6 4.07e+01   7.9 5.99e-02 9.84e-05h  1
 114 -5.7464128e+00 1.40e-01 5.35e+13   3.6 4.19e+02   7.4 1.37e-03 2.18e-05h  6
 115 -5.7621594e+00 1.40e-01 7.44e+13   3.6 2.59e+02   6.9 2.08e-03 5.30e-06h  9
 116r-5.7621594e+00 1.40e-01 1.00e+03   3.6 0.00e+00    -  0.00e+00 2.81e-07R  9
 117r-5.5598909e+00 1.04e+01 6.37e+01  -2.5 1.54e+02    -  9.65e-01 1.63e-01f  1
 118r-4.1889478e+00 5.35e+00 2.33e+01  -2.6 6.48e+00    -  4.78e-01 9.61e-01f  1
 119r-2.1087729e+00 3.57e+00

Output statements

In [8]:
print("\n" + "="*40)
print("       Grass'S GLIDER DESIGN METRICS     ")
print("="*40)
print(f"Optimization Status : {sol.stats()['return_status']}")
print(f"Total Glider Mass   : {sol.value(weight) * 1000:.2f} grams")
print(f"Design Velocity     : {sol.value(V):.2f} m/s")
print("-"*40)
print(f"Main Wing Area (S)  : {sol.value(S):.4f} m^2")
print(f"Aspect Ratio (AR)   : {sol.value(AR):.2f}")
print(f"Total Wingspan (b)  : {sol.value(b_total):.2f} m")
print(f"Wing Chord (c)      : {sol.value(c)*1000:.1f} mm")
print(f"Wing Dihedral       : {sol.value(dihedral):.1f} degrees")
print(f"Wing Twist          : {sol.value(twist):.4f} degrees")
print(f"Glide Angle          : {sol.value(glide_angle):.1f} degrees")
print("-"*40)
print(f"Horizontal Tail Area: {sol.value(S_h):.4f} m^2")
print(f"Vertical Tail Area  : {sol.value(S_v):.4f} m^2")

print(f"Horizontal Tail AR: {sol.value(AR_h):.4f} m^2")
print(f"Horizontal Tail b: {sol.value(b_h * 2):.4f} m")
print(f"Horizontal Tail c: {sol.value(c_h):.4f} m")

print(f"Vertical Tail AR: {sol.value(AR_v):.4f} m^2")
print(f"Vertical Tail b: {sol.value(b_v):.4f} m")
print(f"Vertical Tail c: {sol.value(c_v):.4f} m")

print(f"Tail Boom Length    : {sol.value(H_loc):.2f} m")
print(f"Tail Boom Length    : {sol.value(H_loc_v):.2f} m")
print(f"decalage    : {sol.value(alpha_t):.4f} degrees")

print("-"*40)
print(f"Aerodynamic Lift    : {sol.value(L):.3f} N")
print(f"Required Weight Lift: {sol.value(weight)*9.81:.3f} N")
print(f"Operating CL        : {sol.value(CL):.3f}")
print("-"*40)
print(f"Horiz. Vol. Coeff.  : {sol.value(hor_vol_coef):.3f}  (Target: 0.3 - 0.6)")
print(f"Vert. Vol. Coeff.   : {sol.value(ver_vol_coef):.3f}  (Target: 0.02 - 0.05)")
print(f"Spiral Criterion (B): {sol.value(spiral):.3f}")
# print(sol.value(CL_h))
# print(sol.value(CL))
# print(sol.value(twist))
# print(sol.value( a0 / (1 + a0 / (pi * AR * e)) * twist))

# print(sol.value(max_deflection(N * weight * 9.81, AR_h, S_h, E, I_formula = I_formula, tau=0.002)))
# print(sol.value(b_h))

# print(sol.value(np))
# print(sol.value(COM))


       Grass'S GLIDER DESIGN METRICS     
Optimization Status : Solve_Succeeded
Total Glider Mass   : 24.62 grams
Design Velocity     : 7.00 m/s
----------------------------------------
Main Wing Area (S)  : 0.0182 m^2
Aspect Ratio (AR)   : 5.00
Total Wingspan (b)  : 0.30 m
Wing Chord (c)      : 60.4 mm
Wing Dihedral       : 10.0 degrees
Wing Twist          : 0.0457 degrees
Glide Angle          : 6.3 degrees
----------------------------------------
Horizontal Tail Area: 0.0052 m^2
Vertical Tail Area  : 0.0031 m^2
Horizontal Tail AR: 10.0000 m^2
Horizontal Tail b: 0.2274 m
Horizontal Tail c: 0.0227 m
Vertical Tail AR: 3.0000 m^2
Vertical Tail b: 0.0971 m
Vertical Tail c: 0.0324 m
Tail Boom Length    : -0.05 m
Tail Boom Length    : 0.00 m
decalage    : 0.1348 degrees
----------------------------------------
Aerodynamic Lift    : 0.240 N
Required Weight Lift: 0.242 N
Operating CL        : 0.199
----------------------------------------
Horiz. Vol. Coeff.  : 0.300  (Target: 0.3 - 0.6)
Vert

In [9]:
airplane = asb.Airplane(
    name="Peter's Glider",
    xyz_ref=[0, 0, 0],  
    wings=[
        asb.Wing(
            name="Main Wing",
            symmetric=True,  
            xsecs=[  
                asb.WingXSec(  
                    xyz_le=[0, 0, 0],  
                    chord=c,
                    twist=twist,  
                    airfoil=wing_airfoil,  
                ),
                asb.WingXSec(  
                    xyz_le=[0, b, b * dihedral * pi / 180],
                    chord=c,
                    twist=twist,
                    airfoil=wing_airfoil,
                ),
            ],
        ),
        asb.Wing(
            name="Horizontal Stabilizer",
            symmetric=True,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_h,
                    twist=alpha_t,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, b_h, 0], chord=c_h, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc, 0, 0]),
        asb.Wing(
            name="Vertical Stabilizer",
            symmetric=False,
            xsecs=[
                asb.WingXSec(
                    xyz_le=[0, 0, 0],
                    chord=c_v,
                    twist=0,
                    airfoil=tail_airfoil,
                ),
                asb.WingXSec(
                    xyz_le=[0, 0, b_v], chord=c_v, twist=0, airfoil=tail_airfoil
                ),
            ],
        ).translate([H_loc_v, 0, 0]),
    ],
)

optimized_airplane = sol.value(airplane)

# This will open up an interactive browser window showing your full 3D layout!
optimized_airplane.draw()

Widget(value='<iframe src="http://localhost:36429/index.html?ui=P_0x705fec36aba0_0&reconnect=auto" class="pyvi…

PolyData (0x706006bb8c40)
  N Cells:    725
  N Points:   730
  N Strips:   0
  X Bounds:   -5.439e-02, 6.035e-02
  Y Bounds:   -1.509e-01, 1.509e-01
  Z Bounds:   -3.121e-04, 9.707e-02
  N Arrays:   0

In [110]:
opti.debug.show_infeasibilities()



Violated constraints (tol 0), in order of declaration:
------- i = 1/70 ------ 
0.4 <= 0.4 <= inf (viol 9.97921e-09)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 4/70 ------ 
-inf <= 10 <= 10 (viol 9.84638e-08)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 7/70 ------ 
0 <= -0.0223912 <= inf (viol 0.0223912)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  super().subject_to(constraint)
------- i = 13/70 ------ 
-inf <= 3 <= 3 (viol 2.99504e-08)
Opti constraint of shape 1x1, called from /home/poogle/dbf2027/.venv/lib/python3.12/site-packages/aerosandbox/optimization/opti.py:424 in subject_to
  